# Know-Your-Rights — Legal RAG Database Builder (Kaggle, T4×2)
Builds one unified, citation-ready vector database of **central** Indian law for a public-facing
chatbot. End-to-end here on Kaggle: **load → verify → clean → currency-filter → chunk → enrich
(vLLM on 2×T4) → embed (multi-GPU) → LanceDB → evaluate → download**. Continue locally afterward.

### Sources (all central)
| Source | Rows | Shape | Notes |
|---|---|---|---|
| `mratanusarkar/Indian-Laws` | 34,244 | `act_title, section, law` | ~1,021 acts; **drop repealed IPC/CrPC/Evidence**; some 80k-word schedules |
| `Sharathhebbar24/Indian-Constitution` | 454 | `article_id, article_desc` | Preamble added manually |
| **BNS 2023** (your CSV) | 358 | `Chapter, …, Section, Description` | replaces IPC; `effective 2024-07-01` |
| **BNSS 2023** (your CSV) | 531 | same as BNS | replaces CrPC; official India Code text; `effective 2024-07-01` |
| **BSA 2023** (your CSV) | 170 | same as BNS | replaces Evidence Act; official India Code (as on Oct 2025) |

### Currency is explicit (your requirement)
Every row carries `act_year`, `status` (in_force / omitted / repealed), `effective_date`, and a
`source_snapshot` (data vintage). The LLM can therefore say *"this is the Act as enacted in 2019,
still in force"* and caveat anything newer than the snapshot — recent-amendment checking is done
LLM-side with web tools later. We remove the laws we **know** are dead (the colonial criminal codes).

### Models (fixed for later reuse — the DB is tied to the embedder)
- Embedding (locked, the DB is tied to it): **BAAI/bge-m3** (dim 1024, 8192-token context, multilingual)
- Reranker (eval/query-time, swappable): **Alibaba-NLP/gte-reranker-modern-bert-base** (English, long-context)
- Enrichment LLM: **Qwen2.5-7B-Instruct** via vLLM, `tensor_parallel_size=2` (uses both T4s)

### Kaggle setup (do this first)
Settings → **Accelerator = GPU T4 ×2**, **Internet = On**. Add your BNS CSV via **+ Add Input →
Datasets** (and BNSS/BSA if you have them). The notebook auto-finds them under `/kaggle/input/`.

## 0. Mode & install
This notebook is **load-or-build**. If a prebuilt `legal_db_bundle.zip` (or its unzipped contents) is
attached under `/kaggle/input/`, it **loads** the database and jumps straight to querying (§9–§10).
Otherwise it **builds** from scratch (HF datasets + your BNS/BNSS/BSA CSVs). Flip `FORCE_BUILD=True` to
rebuild even if a bundle is attached. Building needs **GPU T4×2 + Internet On**; loading is light
(no source CSVs, no vLLM).

In [1]:
import os, glob
FORCE_BUILD = False   # set True to rebuild even when a prebuilt bundle is attached
def _find_prebuilt():
    for z in glob.glob("/kaggle/input/**/legal_db_bundle.zip", recursive=True): return ("zip", z)
    for p in glob.glob("/kaggle/input/**/chunks_metadata.parquet", recursive=True): return ("dir", os.path.dirname(p))
    for d in glob.glob("/kaggle/input/**/legal_db", recursive=True):
        if os.path.isdir(d): return ("dir", os.path.dirname(d))
    return (None, None)
PREBUILT_KIND, PREBUILT_PATH = _find_prebuilt()
BUILD = FORCE_BUILD or (PREBUILT_KIND is None)
print("MODE:", "BUILD FROM SCRATCH" if BUILD else f"LOAD PREBUILT ({PREBUILT_KIND} @ {PREBUILT_PATH})")

import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)
pip("-U", "sentence-transformers", "lancedb")      # query + vector store (both modes)
# Remove torchcodec: newest sentence-transformers imports it (audio/video, unused) but Kaggle's
# prebuilt torchcodec is ABI-incompatible with this torch and raises a RuntimeError ST can't catch.
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchcodec"], check=False)
pip("pydantic>=2", "pandas", "tqdm", "pyarrow")    # both modes
if BUILD:                                          # heavy, build-only:
    pip("datasets", "vllm==0.22.0")
    # FlashInfer's attention kernel crashes at runtime on T4 (compute 7.5); removing it makes vLLM
    # auto-select the Triton backend. vLLM treats FlashInfer as optional, so this is safe.
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","flashinfer-python","flashinfer"], check=False)
print("installs done | BUILD =", BUILD)

MODE: LOAD PREBUILT (dir @ /kaggle/input/datasets/damnyadav/legaldata)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.7/336.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
installs done | BUILD = False


## 1. Config — every knob here

In [2]:
import os, glob
# ---------- paths ----------
WORK       = "/kaggle/working"
DB_PATH    = f"{WORK}/legal_db"
TABLE      = "laws"
ENRICH_CACHE = f"{WORK}/enrichment_cache.json"
BUNDLE     = f"{WORK}/legal_db_bundle.zip"

# auto-find uploaded sanhita CSVs anywhere under /kaggle/input
def find_csv(*names):
    for n in names:
        hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        if hits: return hits[0]
    return None
BNS_CSV  = find_csv("bns_sections.csv", "BNS.csv")
BNSS_CSV = find_csv("bnss_sections.csv", "BNSS.csv")   # optional
BSA_CSV  = find_csv("bsa_sections.csv", "BSA.csv")     # optional
print("BNS :", BNS_CSV); print("BNSS:", BNSS_CSV); print("BSA :", BSA_CSV)

# ---------- HF datasets ----------
ACTS_DS  = "mratanusarkar/Indian-Laws"
CONST_DS = "Sharathhebbar24/Indian-Constitution"

# ---------- models ----------
EMBED_MODEL   = "BAAI/bge-m3"                   # LOCKED — multilingual, 8192-ctx, 1024-dim, XLM-R (no custom code)
RERANK_MODEL  = "Alibaba-NLP/gte-reranker-modernbert-base"  # English long-context reranker
ENRICH_MODEL  = "Qwen/Qwen2.5-7B-Instruct"     # Apache-2.0; uses both T4s via TP=2
# lighter/faster fallback: "Qwen/Qwen2.5-1.5B-Instruct" with TP=1
TP            = 2                               # tensor-parallel across the two T4s
EMBED_QUERY_PREFIX = ""                         # gte-v1.5 encodes queries & passages symmetrically (no prefix)

# ---------- enrichment generation ----------
ENRICH_MAX_CHARS  = 1500    # chars of section text the LLM reads (keeps prompts fast)
ENRICH_MAX_LEN    = 2048    # vLLM context
ENRICH_MAX_TOKENS = 320
ENRICH_BATCH      = 2000    # cache checkpoint every N sections
ENRICH_RETRIES    = 3

# ---------- chunking ----------
MAX_WORDS, OVERLAP, MAX_CHUNKS = 480, 80, 25    # cap chunks/section so 80k-word schedules don't explode
MAX_SEQ_LEN = 1024

# ---------- retrieval ----------
FETCH_K, TOP_K, RRF_K = 25, 5, 60
LOW_SCORE = 0.05
MMR_LAMBDA = 0.6            # diversity vs relevance in final ordering

# ---------- citizen-facing category taxonomy (LLM assigns one per section) ----------
CATEGORIES = ["Fundamental Rights","Criminal & Police","Consumer & Services","Employment & Labour",
 "Family & Marriage","Property & Housing","Women & Children","Privacy & Data","Health & Medicine",
 "Education","Environment","Taxation & Finance","Business & Companies","Information & RTI",
 "Civil Procedure & Courts","Transport & Motor","Government & Administration","Other"]

REPEALED_ACTS = {  # colonial criminal codes replaced by BNS/BNSS/BSA on 2024-07-01
 "Indian Penal Code, 1860","Code of Criminal Procedure Act, 1973",
 "Code of Criminal Procedure (Amendment) Act, 1980","Indian Evidence Act, 1872"}

print("config loaded |", EMBED_MODEL, "| enrich:", ENRICH_MODEL, "| TP:", TP)

BNS : /kaggle/input/datasets/nandr39/bharatiya-nyaya-sanhita-dataset-bns/bns_sections.csv
BNSS: /kaggle/input/datasets/damnyadav/legalrights/bnss_sections.csv
BSA : /kaggle/input/datasets/damnyadav/legalrights/bsa_sections.csv
config loaded | BAAI/bge-m3 | enrich: Qwen/Qwen2.5-7B-Instruct | TP: 2


In [3]:
import os, re, gc, json, time, math, zipfile
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
print("Torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  ", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 1, "Enable GPU T4 x2 in Settings"

Torch 2.10.0+cu128 | CUDA True | GPUs 2
   0 Tesla T4
   1 Tesla T4


## 1b. Load prebuilt database (load mode only)
Runs **only** when a bundle is attached. Unzips/locates `legal_db/`, copies it into the writable
working dir (LanceDB needs write access), loads `chunks_metadata.parquet` + `enrichment_cache.json`,
loads the query embedder, and opens the table. After this, §2–§8 are skipped — go straight to §9.
In **build** mode this whole cell is a no-op.

In [4]:
if not BUILD:
    import shutil, zipfile, lancedb
    from sentence_transformers import SentenceTransformer
    if PREBUILT_KIND == "zip":
        ex = f"{WORK}/_prebuilt"
        if os.path.exists(ex): shutil.rmtree(ex)
        with zipfile.ZipFile(PREBUILT_PATH) as z: z.extractall(ex)
        root = ex
    else:
        root = PREBUILT_PATH
    def _find(name, isdir=False):                       # locate a piece anywhere under the input tree
        for q in glob.glob(f"{root}/**/{name}", recursive=True):
            if (os.path.isdir(q) if isdir else os.path.isfile(q)): return q
        return None
    src_db = _find("legal_db", isdir=True); pq = _find("chunks_metadata.parquet"); cj = _find("enrichment_cache.json")
    assert src_db and pq, f"prebuilt bundle incomplete: legal_db={src_db}, parquet={pq}"
    if os.path.abspath(src_db) != os.path.abspath(DB_PATH):    # copy out of read-only /kaggle/input
        if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
        shutil.copytree(src_db, DB_PATH)
    chunks = pd.read_parquet(pq)
    cache  = json.load(open(cj)) if cj else {}
    emb = SentenceTransformer(EMBED_MODEL, trust_remote_code=True); emb.max_seq_length = MAX_SEQ_LEN
    db = lancedb.connect(DB_PATH); tbl = db.open_table(TABLE)
    print(f"LOADED prebuilt -> {tbl.count_rows()} rows | {len(chunks)} chunks | cache {len(cache)} | emb {EMBED_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

LOADED prebuilt -> 38921 rows | 38921 chunks | cache 35200 | emb BAAI/bge-m3


## 2. Load & verify every source
**Build mode only** — §2–§8 are skipped automatically when a prebuilt bundle is loaded.

In [5]:
if BUILD:
    from datasets import load_dataset

    acts = load_dataset(ACTS_DS, split="train").to_pandas()
    print("ACTS :", acts.shape, acts.columns.tolist(), "| distinct acts:", acts["act_title"].nunique())

    const = load_dataset(CONST_DS, split="train").to_pandas()
    print("CONST:", const.shape, const.columns.tolist())

    assert BNS_CSV, "Upload bns_sections.csv as a Kaggle input (+ Add Input -> Datasets)"
    bns = pd.read_csv(BNS_CSV); bns.columns = [re.sub(r"\s+","",c) for c in bns.columns]
    print("BNS  :", bns.shape, bns.columns.tolist())

## 3. Cleaning helpers (evidence-based)
Strip the act-title that repeats at the start of every bare-act section, remove amendment footnote
markers like `4[…]`, pull out embedded `Chapter` headings, normalise whitespace, flag omitted stubs,
and parse a numeric section key + the enactment year.

In [6]:
if BUILD:
    def norm_ws(t):
        return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n{2,}", "\n", str(t).replace("\r\n","\n"))).strip()

    def strip_footnotes(t):                       # 4[(a) ...]  / 1[21. ...]  amendment markers
        t = re.sub(r"\b\d+\[", "", t)
        return t.replace("]", "")

    def extract_chapter(t):                        # "Chapter II Enrolment ..." at section start
        m = re.match(r"\s*Chapter\s+([IVXLC]+)\s+([A-Z][A-Za-z ,&]+?)\s+\d", t)
        if m: return f"Chapter {m.group(1)} - {m.group(2).strip()}", t[m.end()-1:]
        return None, t

    def parse_year(title):
        yrs = re.findall(r"\b(1[5-9]\d\d|20\d\d)\b", str(title))
        return int(yrs[-1]) if yrs else None

    def parse_secnum(s):
        m = re.match(r"\s*(\d+)\s*([A-Z]*)", str(s))
        return (int(m.group(1)) + (ord(m.group(2)[0])-64)/100 if m and m.group(2) else
                (int(m.group(1)) if m else 10**9))

    def short_name(title):
        return re.sub(r",?\s*(1[5-9]\d\d|20\d\d)\s*$", "", str(title)).strip()

    OMIT_RE = re.compile(r"\b(omitted|repealed)\b", re.I)
    def status_of(text):
        t = text.strip()
        return "omitted" if (len(t) < 200 and OMIT_RE.search(t)) else "in_force"

## 4. Map each source to one unified schema
Common columns: `source_type, act_title, act_short_name, act_year, act_number, section_label,
section_num, section_name, chapter, jurisdiction, status, effective_date, full_text, source_snapshot`.

In [7]:
if BUILD:
    UNI = ["source_type","act_title","act_short_name","act_year","act_number","section_label",
           "section_num","section_name","chapter","jurisdiction","status","effective_date",
           "full_text","source_snapshot"]

    # ---- bare central acts ----
    def map_acts(df):
        df = df[~df["act_title"].isin(REPEALED_ACTS)].copy()
        df = df[df["law"].astype(str).str.len() > 1]
        df = df.drop_duplicates(["act_title","section"])
        rows = []
        for _, r in tqdm(df.iterrows(), total=len(df), desc="acts"):
            law = norm_ws(r["law"])
            st  = short_name(r["act_title"])
            law = re.sub(r"^\s*(the\s+)?"+re.escape(r["act_title"])+r"[,.\s]*", "", law, count=1, flags=re.I)
            chap, law = extract_chapter(law)
            law = norm_ws(strip_footnotes(law))
            rows.append(dict(source_type="central_act", act_title=r["act_title"], act_short_name=st,
                act_year=parse_year(r["act_title"]), act_number=None, section_label=str(r["section"]),
                section_num=parse_secnum(r["section"]), section_name="", chapter=chap,
                jurisdiction="central", status=status_of(law), effective_date=None,
                full_text=law, source_snapshot="Central Acts (mratanusarkar/Indian-Laws), ~Jan 2024"))
        return pd.DataFrame(rows)

    # ---- constitution ----
    PART_RANGES=[(1,4,"Union & Territory"),(5,11,"Citizenship"),(12,35,"Fundamental Rights"),
     (36,51,"Directive Principles"),(51,51,"Fundamental Duties"),(52,151,"The Union"),
     (152,237,"The States"),(239,242,"Union Territories"),(243,243,"Panchayats/Municipalities"),
     (244,244,"Scheduled & Tribal Areas"),(245,263,"Union-State Relations"),
     (264,300,"Finance & Property"),(301,307,"Trade & Commerce"),(308,323,"Services & Tribunals"),
     (324,329,"Elections"),(330,342,"Special Provisions"),(343,351,"Official Language"),
     (352,360,"Emergency"),(361,367,"Miscellaneous"),(368,368,"Amendment"),(369,395,"Transitional")]
    def const_part(n):
        for lo,hi,nm in PART_RANGES:
            if n is not None and lo<=n<=hi: return nm
        return "Constitution"
    def map_const(df):
        pre=("WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN "
             "SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens JUSTICE, LIBERTY, "
             "EQUALITY and FRATERNITY; IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, "
             "1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION.")
        rows=[dict(source_type="constitution",act_title="Constitution of India",act_short_name="Constitution",
            act_year=1950,act_number=None,section_label="Preamble",section_num=0,section_name="Preamble",
            chapter="Preamble",jurisdiction="central",status="in_force",effective_date="1950-01-26",
            full_text=pre,source_snapshot="Constitution of India (with amendments), dataset ~2023")]
        for _, r in df.iterrows():
            m=re.search(r"Article\s+(\d+)([A-Z]*)", str(r["article_id"]))
            num=int(m.group(1)) if m else None; suf=m.group(2) if m else ""
            rows.append(dict(source_type="constitution",act_title="Constitution of India",
                act_short_name="Constitution",act_year=1950,act_number=None,
                section_label=f"{num}{suf}" if num else str(r["article_id"]),
                section_num=(num+(ord(suf[0])-64)/100 if suf else (num or 10**9)),
                section_name="",chapter=const_part(num),jurisdiction="central",
                status="in_force",effective_date=None,full_text=norm_ws(r["article_desc"]),
                source_snapshot="Constitution of India (with amendments), dataset ~2023"))
        return pd.DataFrame(rows)

    # ---- a sanhita CSV (BNS / BNSS / BSA share one shape) ----
    def map_sanhita(df, title, number, eff):
        df = df.copy(); df.columns=[re.sub(r"\s+","",c) for c in df.columns]
        rows=[]
        for _, r in df.iterrows():
            rows.append(dict(source_type="criminal_code",act_title=title,act_short_name=title.split(",")[0],
                act_year=2023,act_number=number,section_label=str(r["Section"]),
                section_num=parse_secnum(r["Section"]),section_name=norm_ws(r.get("Section_name","")),
                chapter=norm_ws(r.get("Chapter_name","")),jurisdiction="central",status="in_force",
                effective_date=eff,full_text=norm_ws(strip_footnotes(str(r["Description"]))),
                source_snapshot=f"{title} — official text, in force {eff}"))
        return pd.DataFrame(rows)

In [8]:
if BUILD:
    parts = [map_acts(acts), map_const(const),
             map_sanhita(bns, "Bharatiya Nyaya Sanhita, 2023", "45 of 2023", "2024-07-01")]
    if BNSS_CSV: parts.append(map_sanhita(pd.read_csv(BNSS_CSV),
                    "Bharatiya Nagarik Suraksha Sanhita, 2023", "46 of 2023", "2024-07-01"))
    if BSA_CSV:  parts.append(map_sanhita(pd.read_csv(BSA_CSV),
                    "Bharatiya Sakshya Adhiniyam, 2023", "47 of 2023", "2024-07-01"))

    uni = pd.concat(parts, ignore_index=True)[UNI]
    uni = uni[uni["full_text"].str.len() > 1].reset_index(drop=True)
    uni["unit_id"] = uni["source_type"]+"|"+uni["act_title"]+"|"+uni["section_label"]
    uni = uni.drop_duplicates("unit_id").reset_index(drop=True)
    print("UNIFIED rows:", len(uni))
    print(uni["source_type"].value_counts().to_string())
    print("\nrepealed criminal codes still present?",
          uni["act_title"].isin(REPEALED_ACTS).sum(), "(expect 0)")
    uni.head(3)

## 5. Chunk
Section is the base unit (= the citation unit). Long sections split with overlap; chunks/section are
capped so the 80k-word schedules don't explode the corpus. **Enrichment runs once per section** (not
per chunk) and every chunk of that section inherits it.

In [9]:
if BUILD:
    def split_words(t, mw=MAX_WORDS, ov=OVERLAP, cap=MAX_CHUNKS):
        w=t.split()
        if len(w)<=mw: return [t]
        out=[]; i=0
        while i<len(w) and len(out)<cap:
            out.append(" ".join(w[i:i+mw])); i+=mw-ov
        return out

    rows=[]
    for _, r in tqdm(uni.iterrows(), total=len(uni), desc="chunking"):
        for j, ch in enumerate(split_words(r["full_text"])):
            d=r.to_dict(); d["chunk_id"]=f'{r["unit_id"]}#{j}'; d["chunk_text"]=ch; rows.append(d)
    chunks = pd.DataFrame(rows)
    print(f"sections {len(uni)} -> chunks {len(chunks)} | longest chunk words:",
          chunks["chunk_text"].str.split().str.len().max())

## 6. Enrichment — strict Pydantic, T4-safe engine
One LLM call per **section** returns lay `questions`, `keywords`, and a citizen-facing `category`
(fixed taxonomy). Output is validated against a Pydantic schema; malformed results are retried (with
temperature) and **never cached**; the gate halts the pipeline if anything is invalid. Resumable.

> **Kaggle T4 fix (baked in).** T4s are compute-7.5. FlashInfer's attention kernel crashes at *runtime*
> on Turing (`BatchPrefillWithPagedKVCache: invalid argument`), and on vLLM 0.22 the old
> `VLLM_ATTENTION_BACKEND` override is gone. So §0 **uninstalls FlashInfer** and vLLM auto-selects the
> **Triton** attention backend, which works on T4 (Torch sampler too). After the engine starts, a
> 1-token **smoke test** runs a real prefill+decode; if any kernel is still unhappy, the cell **falls
> back to plain HuggingFace generation** (no vLLM/FlashInfer at all), so this step completes regardless.
>
> **Cached re-runs skip the model entirely** — after a kernel restart, §6 loads nothing and leaves the
> GPU free for §8. To go faster at slightly lower quality, set `ENRICH_MODEL="Qwen/Qwen2.5-1.5B-Instruct"`
> and `TP=1` in §1.
>
> **Resuming after a timeout:** if a run ended mid-build, download `enrichment_cache.json` from the
> output bundle (or the working dir), then in the new session **Add Input** → your previous notebook
> output (or upload the file as a dataset). The scope cell auto-detects any attached `enrichment_cache.json`
> and seeds the working cache from it, so only the unfinished sections are processed. The gate then
> auto-fills any sections the model never managed to format (a handful at most) so the build completes.

In [10]:
if BUILD:
    from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

    class Enrichment(BaseModel):
        model_config = {"extra":"ignore"}
        questions: list[str] = Field(min_length=1, max_length=8)
        keywords:  list[str] = Field(min_length=3, max_length=15)
        category:  str
        @field_validator("questions","keywords", mode="after")
        @classmethod
        def _clean(cls, v):
            out=[s.strip() for s in v if isinstance(s,str) and s.strip()]
            if not out: raise ValueError("empty after strip")
            return out
        @field_validator("category", mode="after")
        @classmethod
        def _cat(cls, v):
            v=v.strip(); return v if v in CATEGORIES else "Other"
        @model_validator(mode="after")
        def _noleak(self):
            for q in self.questions:
                if re.search(r"\bKEYWORDS?\b\s*[:\-]", q, re.I): raise ValueError("heading leak")
            return self

    # self-test (no model call)
    Enrichment.model_validate({"questions":["can I be arrested?"],
        "keywords":["arrest","liberty","police"],"category":"Criminal & Police"})
    print("schema OK | categories:", len(CATEGORIES))

In [11]:
if BUILD:
    # ---- scope: what still needs enriching? (define BEFORE loading any model) ----
    # RESUME: if a previous run's enrichment_cache.json is attached as a Kaggle input (Add Input → your
    # earlier notebook output / a dataset), copy it into the working cache so we DON'T redo finished work.
    if not os.path.exists(ENRICH_CACHE):
        import glob, shutil
        prior = sorted(glob.glob("/kaggle/input/**/enrichment_cache.json", recursive=True),
                       key=os.path.getsize, reverse=True)   # if several attached, take the most complete
        if prior:
            shutil.copy(prior[0], ENRICH_CACHE)
            print("seeded working cache from prior run:", prior[0])
    units = uni[["unit_id","act_title","section_label","section_name","full_text"]].to_dict("records")
    cache = json.load(open(ENRICH_CACHE)) if os.path.exists(ENRICH_CACHE) else {}
    # redo anything not cached OR previously marked _failed (stragglers get another attempt)
    todo  = [u for u in units if u["unit_id"] not in cache or cache.get(u["unit_id"],{}).get("_failed")]
    print(f"sections {len(units)} | cached {len(cache)} | to enrich {len(todo)}")

    SYS=("You write search metadata for a section of Indian central law, for an app that helps ordinary "
         "citizens. Base everything ONLY on the provided text; never invent legal facts. Reply with a "
         "single JSON object and nothing else: {\"questions\":[3 everyday questions a normal person might "
         "ask that this provision answers],\"keywords\":[6-10 short topic keywords],\"category\":\"one of "
         + " | ".join(CATEGORIES) + "\"}.")

    def prompt_for(u):
        txt=f'{u["act_title"]} — Section {u["section_label"]}'
        if u["section_name"]: txt+=f' ({u["section_name"]})'
        txt+=":\n"+u["full_text"][:ENRICH_MAX_CHARS]
        return [{"role":"system","content":SYS},{"role":"user","content":txt}]

    def parse_one(text):
        s,e=text.find("{"),text.rfind("}")
        if s<0 or e<0: raise ValueError("no json")
        return Enrichment.model_validate(json.loads(text[s:e+1]))

In [12]:
if BUILD:
    # ---- LLM for enrichment. Loads ONLY if something needs enriching (a cached re-run after a kernel
    #      restart is then a GPU no-op, leaving the card free for §8).
    #
    #      Kaggle T4 = compute 7.5. FlashInfer's attention kernel crashes at runtime on Turing, so §0
    #      uninstalls FlashInfer and vLLM auto-selects the Triton attention backend (works on T4); the
    #      Torch sampler is used too. After init we run a 1-token SMOKE TEST so that if any GPU kernel is
    #      still unhappy at runtime, we fall back to plain HuggingFace generation (no vLLM/FlashInfer).
    llm = None; hf = None; BACKEND = None
    if todo:
        import glob
        # make -lcuda resolvable for any CUDA JIT compile (belt-and-suspenders)
        cand = (glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") +
                glob.glob("/usr/local/nvidia/lib64/libcuda.so*") + glob.glob("/usr/lib64/libcuda.so*"))
        if not cand:
            import subprocess
            cand = [l for l in subprocess.run(["find","/usr","-name","libcuda.so*"],
                    capture_output=True, text=True).stdout.split("\n") if l.strip()]
        if cand:
            drv = sorted(cand, key=len)[0]
            for d in ["/usr/local/cuda/lib64/stubs", "/usr/local/cuda/lib64"]:
                try:
                    if os.path.isdir(d) and not os.path.exists(f"{d}/libcuda.so"): os.symlink(drv, f"{d}/libcuda.so")
                except OSError: pass
            os.environ["LIBRARY_PATH"] = "/usr/local/cuda/lib64/stubs:/usr/local/cuda/lib64:" + os.environ.get("LIBRARY_PATH","")
        os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
        os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
        os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(ENRICH_MODEL)
        try:
            from vllm import LLM, SamplingParams
            llm = LLM(model=ENRICH_MODEL, tensor_parallel_size=TP, dtype="half",
                      gpu_memory_utilization=0.92, max_model_len=ENRICH_MAX_LEN,
                      trust_remote_code=True, enforce_eager=True, disable_custom_all_reduce=True)
            # smoke test: force one real prefill+decode so a runtime kernel failure is caught HERE, not
            # mid-enrichment (this is exactly where FlashInfer died on T4).
            _ = llm.generate(["ping"], SamplingParams(temperature=0, max_tokens=1), use_tqdm=False)
            def gen(items, temperature):
                prompts=[tok.apply_chat_template(prompt_for(u), tokenize=False, add_generation_prompt=True) for u in items]
                sp=SamplingParams(temperature=temperature, top_p=0.9, max_tokens=ENRICH_MAX_TOKENS)
                return [o.outputs[0].text for o in llm.generate(prompts, sp, use_tqdm=False)]
            BACKEND="vllm"; print("vLLM ready (Triton attention, smoke test passed):", ENRICH_MODEL)
        except Exception as e:
            print("vLLM unusable on this GPU:", str(e)[:200])
            print("falling back to HuggingFace generation (no vLLM/FlashInfer) on", ENRICH_MODEL)
            try: del llm
            except NameError: pass
            llm = None; gc.collect()
            try: torch.cuda.empty_cache()
            except Exception: pass
            from transformers import AutoModelForCausalLM
            if tok.pad_token is None: tok.pad_token = tok.eos_token
            tok.padding_side = "left"
            hf = AutoModelForCausalLM.from_pretrained(ENRICH_MODEL, torch_dtype=torch.float16,
                                                      device_map="auto", trust_remote_code=True).eval()
            def gen(items, temperature, _micro=24):
                outs=[]
                for k in tqdm(range(0,len(items),_micro), desc="hf-gen", leave=False):
                    pr=[tok.apply_chat_template(prompt_for(u), tokenize=False, add_generation_prompt=True)
                        for u in items[k:k+_micro]]
                    enc=tok(pr, return_tensors="pt", padding=True, truncation=True, max_length=ENRICH_MAX_LEN).to("cuda")
                    with torch.no_grad():
                        out=hf.generate(**enc, max_new_tokens=ENRICH_MAX_TOKENS, do_sample=(temperature>0),
                                        temperature=max(temperature,1e-5), top_p=0.9, pad_token_id=tok.pad_token_id)
                    outs.extend(tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True))
                return outs
            BACKEND="hf"; print("HuggingFace generator ready:", ENRICH_MODEL)
    else:
        print("all sections already enriched — skipping model load; GPU left free for §8.")

In [13]:
if BUILD:
    # resumable, validated enrichment (uses todo / cache / gen from the cells above)
    for i in tqdm(range(0, len(todo), ENRICH_BATCH), desc="enrich-batches"):
        batch = todo[i:i+ENRICH_BATCH]
        pending = {u["unit_id"]: u for u in batch}
        for attempt in range(ENRICH_RETRIES):
            if not pending: break
            items = list(pending.values())
            texts = gen(items, temperature=0.0 if attempt==0 else 0.6)
            for u, t in zip(items, texts):
                try:
                    cache[u["unit_id"]] = parse_one(t).model_dump()
                    pending.pop(u["unit_id"], None)
                except (ValidationError, json.JSONDecodeError, ValueError):
                    pass
        for uid in pending: cache[uid] = {"_failed": True}     # marked, caught by the gate
        json.dump(cache, open(ENRICH_CACHE,"w"), ensure_ascii=False)
    print("enrichment pass done | cached:", len(cache))

In [14]:
if BUILD:
    # ---- GATE: every section must end with a schema-valid enrichment ----
    # missing / invalid  => the run-cell didn't finish or a cache entry is corrupt -> re-run the run-cell.
    # failed (a handful) => the model never produced valid JSON for these even after retries. Rather than
    #   block the whole build, fill them with a minimal valid record (their full_text is still embedded and
    #   fully searchable via dense + BM25) and list them so you can review later if you wish.
    missing=[uid for uid in uni["unit_id"] if uid not in cache]
    invalid=[]
    for uid in uni["unit_id"]:
        e=cache.get(uid)
        if e and not e.get("_failed"):
            try: Enrichment.model_validate(e)
            except ValidationError: invalid.append(uid)
    failed=[uid for uid in uni["unit_id"] if cache.get(uid,{}).get("_failed")]
    print(f"missing {len(missing)} | failed {len(failed)} | invalid {len(invalid)}")

    if missing or invalid:
        for uid in invalid: cache.pop(uid, None)            # clear corrupt entries so the re-run retries them
        json.dump(cache, open(ENRICH_CACHE,"w"), ensure_ascii=False)
        raise RuntimeError(f"ENRICHMENT INCOMPLETE — {len(missing)} missing, {len(invalid)} invalid. "
                           "Re-run the run-cell above (it resumes), then re-run this gate.")

    if failed:
        rows={u["unit_id"]: u for u in units}
        for uid in failed:
            r=rows.get(uid, {})
            kws=[w.lower() for w in re.findall(r"[A-Za-z]{4,}", (r.get("section_name") or r.get("act_title") or ""))][:8]
            while len(kws)<3: kws.append("law")
            cache[uid]={"questions":[f"What does this section of {r.get('act_title','this Act')} provide?"],
                        "keywords":kws[:15], "category":"Other", "_autofilled":True}
        json.dump(cache, open(ENRICH_CACHE,"w"), ensure_ascii=False)
        print(f"NOTE: auto-filled minimal enrichment for {len(failed)} section(s) the model couldn't format:")
        for uid in failed: print("   ", uid)

    print("GATE PASSED — all", len(uni), "sections have a valid enrichment.")
    for s in ["constitution|Constitution of India|21","criminal_code|Bharatiya Nyaya Sanhita, 2023|63"]:
        if s in cache: print(" ", s, "->", cache[s])

In [15]:
if BUILD:
    # free the GPU before embedding. If memory is still held, Restart kernel & Run All: enrichment is
    # cached, so §6 skips the model load and §8 runs on a free GPU.
    for _n in ("llm","hf"):
        try:
            if globals().get(_n) is not None: del globals()[_n]
        except Exception: pass
    gc.collect()
    try: torch.cuda.empty_cache()
    except Exception: pass
    print("GPU released (or Restart kernel & Run All if §8 OOMs — enrichment is cached)")

## 7. Build `embed_text` and the citation string

In [16]:
if BUILD:
    def citation(r):
        if r["source_type"]=="constitution":
            base=f'Article {r["section_label"]}, Constitution of India'
        else:
            base=f'Section {r["section_label"]}, {r["act_title"]}'
        if r["effective_date"]: base+=f' (in force from {r["effective_date"]})'
        return base

    def embed_text(r):
        e=cache.get(r["unit_id"], {})
        head=f'{r["act_title"]} — Section {r["section_label"]}'
        if r["section_name"]: head+=f' ({r["section_name"]})'
        parts=[head]+e.get("questions",[])
        if e.get("keywords"): parts.append(" ".join(e["keywords"]))
        parts.append(r["chunk_text"])
        return "\n".join(parts)

    chunks["category"]   = chunks["unit_id"].map(lambda u: cache.get(u,{}).get("category","Other"))
    chunks["citation"]   = chunks.apply(citation, axis=1)
    chunks["embed_text"] = chunks.apply(embed_text, axis=1)
    print(chunks["category"].value_counts().to_string())
    print("\nsample embed_text:\n", chunks.loc[chunks.act_title=="Constitution of India","embed_text"].iloc[0][:300])

## 8. Embed on both T4s → LanceDB
`sentence-transformers` multi-process pool splits the corpus across `cuda:0` and `cuda:1`. Vectors are
L2-normalised. Then a LanceDB table + BM25 FTS for hybrid retrieval.

> If this cell OOMs because vLLM didn't fully release the GPUs, **Restart kernel → Run All**: §6 skips
> the model load when enrichment is cached, so both T4s are free here.

In [17]:
if BUILD:
    from sentence_transformers import SentenceTransformer
    import lancedb

    emb = SentenceTransformer(EMBED_MODEL, trust_remote_code=True)
    emb.max_seq_length = MAX_SEQ_LEN   # bge-m3 supports up to 8192; 1024 covers ~99% of chunks
    texts = chunks["embed_text"].tolist()

    devs = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
    print("embedding on", devs, "for", len(texts), "chunks")
    if len(devs) > 1:
        pool = emb.start_multi_process_pool(target_devices=devs)
        vecs = emb.encode_multi_process(texts, pool, batch_size=64)   # bge-m3 is large; 64 keeps T4 in memory
        emb.stop_multi_process_pool(pool)
    else:
        vecs = emb.encode(texts, batch_size=64, show_progress_bar=True)
    vecs = np.asarray(vecs, dtype="float32")
    vecs /= (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)   # version-safe normalize
    print("vectors:", vecs.shape)

In [18]:
if BUILD:
    import shutil
    if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
    db = lancedb.connect(DB_PATH)
    recs=[]
    for r, v in zip(chunks.to_dict("records"), vecs):
        r=dict(r); r["vector"]=v.tolist(); recs.append(r)
    tbl = db.create_table(TABLE, data=recs, mode="overwrite")
    tbl.create_fts_index("embed_text", use_tantivy=False, replace=True)
    print(f"LanceDB built: {tbl.count_rows()} chunks | dim {vecs.shape[1]} | at {DB_PATH}")

## 9. Retrieval — hybrid + rerank + MMR (diversity) + citation
Dense (gte) + BM25 fused by RRF, cross-encoder rerank, then MMR so the top-k spans different acts
rather than near-duplicate clauses. Returns the citation + currency fields for the chatbot.

In [19]:
from transformers import AutoTokenizer as ATok, AutoModelForSequenceClassification as ASeq
_rt=_rm=None
def reranker():
    global _rt,_rm
    if _rm is None:
        for name in [RERANK_MODEL, "BAAI/bge-reranker-base"]:   # fall back if ModernBERT won't load here
            try:
                _rt=ATok.from_pretrained(name, trust_remote_code=True)
                _rm=ASeq.from_pretrained(name, trust_remote_code=True).to("cuda:0").eval()
                print("reranker:", name); break
            except Exception as ex:
                print("reranker load failed for", name, "->", str(ex)[:140]); _rt=_rm=None
        if _rm is None: raise RuntimeError("no reranker could be loaded")
    return _rt,_rm

def _rr_maxlen(m):                          # cap at the RERANKER's own context, not the embedder's
    cfg=m.config
    mp=int(getattr(cfg,"max_position_embeddings",512) or 512)
    if getattr(cfg,"model_type","") in ("xlm-roberta","roberta","camembert"): mp-=2  # RoBERTa pos offset
    return max(8, min(mp, MAX_SEQ_LEN))

def rr_scores(q, docs):
    t,m=reranker()
    ml=_rr_maxlen(m)
    with torch.no_grad():
        enc=t([[q,d] for d in docs], padding=True, truncation=True, max_length=ml, return_tensors="pt")
        enc.pop("token_type_ids", None)     # XLM-R / RoBERTa rerankers don't use segment ids
        enc={k:v.to("cuda:0") for k,v in enc.items()}
        return torch.sigmoid(m(**enc).logits.view(-1)).float().cpu().numpy()

def rrf(lists, k=RRF_K):
    s={}
    for ranks in lists:
        for p,cid in enumerate(ranks): s[cid]=s.get(cid,0.)+1./(k+p+1)
    return sorted(s, key=s.get, reverse=True)

def mmr(cand_vecs, base, lam=MMR_LAMBDA, k=TOP_K):
    chosen=[]; rest=list(range(len(base)))
    while rest and len(chosen)<k:
        if not chosen: j=int(np.argmax(base)); chosen.append(j); rest.remove(j); continue
        best,bj=-1e9,rest[0]
        for i in rest:
            div=max(float(cand_vecs[i]@cand_vecs[c]) for c in chosen)
            val=lam*base[i]-(1-lam)*div
            if val>best: best,bj=val,i
        chosen.append(bj); rest.remove(bj)
    return chosen

def search(query, fetch=FETCH_K, top_k=TOP_K):
    qv=emb.encode([EMBED_QUERY_PREFIX+query], normalize_embeddings=True)[0].astype("float32")
    dense=tbl.search(qv).limit(fetch).to_pandas()["chunk_id"].tolist()
    try: fts=tbl.search(query, query_type="fts").limit(fetch).to_pandas()["chunk_id"].tolist()
    except Exception: fts=[]
    fused=rrf([dense,fts])[:fetch]
    cand=chunks[chunks.chunk_id.isin(fused)].drop_duplicates("unit_id").copy()
    if cand.empty: return cand
    cand["score"]=rr_scores(query, cand["chunk_text"].tolist())
    cand=cand.sort_values("score",ascending=False).head(min(12,len(cand)))
    cv=emb.encode(cand["chunk_text"].tolist(), normalize_embeddings=True).astype("float32")
    order=mmr(cv, cand["score"].to_numpy(), k=top_k)
    out=cand.iloc[order]
    return out[["citation","category","act_year","status","effective_date","score",
                "source_snapshot","full_text"]]

search("can the police arrest me without telling me why", top_k=4)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

reranker: Alibaba-NLP/gte-reranker-modernbert-base


W0602 07:16:56.458000 58 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


,citation,category,act_year,status,effective_date,score,source_snapshot,full_text
38207,"Section 35, Bharatiya Nagarik Suraksha Sanhita...",Criminal & Police,2023.0,in_force,2024-07-01,0.880832,"Bharatiya Nagarik Suraksha Sanhita, 2023 — off...",(1) Any police officer may without an order fr...
27829,"Section 29, Northern Indian Ferries Act, 1878",Criminal & Police,1878.0,in_force,None,0.841906,"Central Acts (mratanusarkar/Indian-Laws), ~Jan...",29. Power to arrest without warrant.\nThe poli...
28276,"Section 13, Passports Act, 1967",Criminal & Police,1967.0,in_force,None,0.861846,"Central Acts (mratanusarkar/Indian-Laws), ~Jan...",13. Power to arrest\n(1) Any officer of custom...
37321,"Article 22, Constitution of India",Criminal & Police,1950.0,in_force,None,0.832129,"Constitution of India (with amendments), datas...",Protection against arrest and detention in cer...


## 10. Evaluation — coverage across domains + abstention + diversity

In [24]:
GOLD = {  # query -> any acceptable citation substring
 "right to equality before the law":"Article 14",
 "can the police arrest me without telling me why":"Bharatiya Nyaya",   # or BNSS once added
 "freedom to practise my religion":"Article 25",
 "right to life and personal liberty":"Article 21",
 "what is the punishment for cheating":"Bharatiya Nyaya",
 "my consumer complaint against a defective product":"Consumer",
 "how do I file an RTI request for information":"Right to Information",
 "maternity leave entitlement at work":"Maternity",
 "protection from domestic violence":"Domestic Violence",
 "is untouchability banned":"Article 17",
}
STRESS=["my landlord in Mumbai won't return my deposit",      # state subject -> expect abstain
        "is there a constitutional right to privacy",         # jurisprudential
        "rules for my housing society in Karnataka"]          # state subject

hits=rr=0; rows=[]
for q,want in tqdm(GOLD.items(), desc="GOLD"):
    res=search(q); cits=res["citation"].tolist() if len(res) else []
    rank=next((i+1 for i,c in enumerate(cits) if want.lower() in c.lower()), None)
    hits+=rank is not None; rr+=1/rank if rank else 0
    rows.append(f'{"OK " if rank else "MISS"} | want "{want}" | top: {cits[0] if cits else "-"}')
print("\n".join(rows))
print(f"\nRecall@{TOP_K}: {hits}/{len(GOLD)} = {hits/len(GOLD):.0%} | MRR {rr/len(GOLD):.3f}")

GOLD:   0%|          | 0/10 [00:00<?, ?it/s]

OK  | want "Article 14" | top: Article 14, Constitution of India
MISS | want "Bharatiya Nyaya" | top: Section 35, Bharatiya Nagarik Suraksha Sanhita, 2023 (in force from 2024-07-01)
OK  | want "Article 25" | top: Article 25, Constitution of India
OK  | want "Article 21" | top: Article 21, Constitution of India
OK  | want "Bharatiya Nyaya" | top: Section 108, Delhi Police Act, 1978
OK  | want "Consumer" | top: Section 83, Consumer Protection Act, 2019
OK  | want "Right to Information" | top: Section 6, Right to Information Act, 2005
OK  | want "Maternity" | top: Section 79, Factories Act, 1948
OK  | want "Domestic Violence" | top: Section 1, Protection of Women from Domestic Violence Act, 2005
OK  | want "Article 17" | top: Article 17, Constitution of India

Recall@5: 9/10 = 90% | MRR 0.783


In [21]:
print("STRESS (low top-score => abstain & let the LLM web-search):\n")
for q in STRESS:
    res=search(q, top_k=3)
    if res.empty: print(f"Q: {q}\n  (none)\n"); continue
    top=res["score"].iloc[0]
    print(f'Q: {q}  {"<-- ABSTAIN" if top<LOW_SCORE else ""}')
    for _,r in res.iterrows(): print(f'   {r["citation"]}  [{r["category"]}]  {r["score"]:.3f}')
    print()
print("\nDiversity — distinct acts in top-5 for a broad query:")
r=search("what are my fundamental rights as a citizen", top_k=5)
print(r["citation"].tolist())

STRESS (low top-score => abstain & let the LLM web-search):

Q: my landlord in Mumbai won't return my deposit  
   Section 33, Maharashtra Rent Control Act, 1999  [Property & Housing]  0.816
   Section 4, Banning of Unregulated Deposit Schemes Act, 2019  [Consumer & Services]  0.760
   Section 18, Delhi Rent Act, 1995  [Property & Housing]  0.815

Q: is there a constitutional right to privacy  
   Section 79, Bharatiya Nyaya Sanhita, 2023 (in force from 2024-07-01)  [Women & Children]  0.763
   Section 15, Indian Easements Act, 1882  [Property & Housing]  0.746
   Section 21, Constitution of India, 1949  [Fundamental Rights]  0.762

Q: rules for my housing society in Karnataka  
   Section 206, Cooperative Societies Act, 2008  [Other]  0.878
   Section 5, National Institute of Mental Health and Neuro Sciences, Bangalore Act, 2012  [Health & Medicine]  0.786
   Section 5, Delhi Apartment Ownership Act, 1986  [Property & Housing]  0.819


Diversity — distinct acts in top-5 for a broad qu

## 11. Package & download
Zips the LanceDB folder + enrichment cache into `/kaggle/working/legal_db_bundle.zip`. Download it
from the **Output** panel on the right (or **Save Version** → the file lands in the run's output).
Locally, query it with the **same** embedding model, `BAAI/bge-m3` (no query prefix), reusing the
§9 `search()` logic. The reranker is independent of the embedder and can be swapped freely.

In [27]:
import json, os, zipfile
# load mode holds the cache only in memory -> write it back before zipping
json.dump(cache, open(ENRICH_CACHE, "w"), ensure_ascii=False)
# cleaned chunks (post de-dup), vector column dropped (it lives in the LanceDB table)
chunks.drop(columns=["vector"], errors="ignore").to_parquet(f"{WORK}/chunks_metadata.parquet")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(DB_PATH):
        for f in files:
            fp = os.path.join(root, f)
            z.write(fp, os.path.relpath(fp, WORK))          # store as legal_db/...
    z.write(ENRICH_CACHE, "enrichment_cache.json")
    z.write(f"{WORK}/chunks_metadata.parquet", "chunks_metadata.parquet")
print(f"bundle: {BUNDLE} ({os.path.getsize(BUNDLE)/1e6:.1f} MB)")
print("contains: legal_db/ (LanceDB) + enrichment_cache.json + chunks_metadata.parquet")
print(f"table rows: {tbl.count_rows()} | chunks: {len(chunks)} | cache entries: {len(cache)}")

bundle: /kaggle/working/legal_db_bundle.zip (266.8 MB)
contains: legal_db/ (LanceDB) + enrichment_cache.json + chunks_metadata.parquet
table rows: 38890 | chunks: 38890 | cache entries: 35200
